In [ ]:
# Install necessary libraries (run this cell in Colab)
!pip install gradio diffusers transformers accelerate Pillow numpy librosa soundfile opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 128.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12

In [ ]:
import gradio as gr
from PIL import Image
import numpy as np
import librosa
import soundfile as sf
import cv2
import os
from typing import List

In [ ]:
# --- 1. Game Content Generation ---

def generate_game_asset(prompt: str):
    """Generates a 2D game asset using a diffusion model."""
    try:
        from diffusers import StableDiffusionPipeline
        import torch

        # Load pre-trained Stable Diffusion model
        model_id = "runwayml/stable-diffusion-v1-5"
        pipeline = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
        pipeline = pipeline.to("cuda" if torch.cuda.is_available() else "cpu")

        # Generate the image
        image = pipeline(prompt).images[0]
        return image
    except Exception as e:
        return f"Error generating image: {e}. Make sure you have a GPU runtime in Colab for better performance."

def generate_procedural_level(width: int = 32, height: int = 32, density: float = 0.4, iterations: int = 5):
    """Generates a simple procedural level using cellular automata."""
    grid = np.random.rand(height, width) < density
    for _ in range(iterations):
        new_grid = np.copy(grid)
        for i in range(height):
            for j in range(width):
                live_neighbors = 0
                for x in range(max(0, i - 1), min(height, i + 2)):
                    for y in range(max(0, j - 1), min(width, j + 2)):
                        if (x, y) != (i, j) and grid[x, y]:
                            live_neighbors += 1
                if grid[i, j]:
                    if live_neighbors < 2 or live_neighbors > 3:
                        new_grid[i, j] = False
                else:
                    if live_neighbors == 3:
                        new_grid[i, j] = True
        grid = new_grid
    # Convert to an image for display
    level_image = np.uint8(grid * 255)
    return Image.fromarray(level_image)

In [ ]:
# --- 2. Music Composition ---

def generate_melody(genre: str = "Ambient", tempo: int = 120, length_seconds: int = 10):
    """Generates a simple melody (placeholder)."""
    # In a real project, you would use a music generation model here.
    # For simplicity, we'll create a sine wave.
    sample_rate = 44100
    frequency = 440  # A4 note
    t = np.linspace(0, length_seconds, int(sample_rate * length_seconds), False)
    note = 0.5 * np.sin(2 * np.pi * frequency * t)
    return sample_rate, note

def generate_beat(tempo: int = 120, length_seconds: int = 10):
    """Generates a simple beat (placeholder)."""
    # In a real project, you would use a beat generation model here.
    # For simplicity, we'll create a basic kick drum pattern.
    sample_rate = 44100
    kick_frequency = 60
    kick_duration = 0.1
    samples_per_beat = int(60 / tempo * sample_rate)
    kick_samples = int(kick_duration * sample_rate)
    beat = np.zeros(int(length_seconds * sample_rate))

    for i in range(0, len(beat), samples_per_beat):
        kick = 0.8 * np.sin(2 * np.pi * kick_frequency * np.linspace(0, kick_duration, kick_samples, False))
        beat[i:i + kick_samples] = kick

    return sample_rate, beat

def save_audio(samplerate, data, filename="output.wav"):
    """Saves the generated audio to a WAV file."""
    sf.write(filename, data, samplerate)
    return filename


In [ ]:
# --- 3. Video Synthesis ---

def generate_sunset_animation(duration_seconds: int = 10, num_frames: int = 30):
    """Generates a simple sunset animation (placeholder using image generation)."""
    try:
        from diffusers import StableDiffusionPipeline
        import torch
        from PIL import Image

        model_id = "runwayml/stable-diffusion-v1-5"
        pipeline = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
        pipeline = pipeline.to("cuda" if torch.cuda.is_available() else "cpu")

        frames: List[Image.Image] = []
        for i in range(num_frames):
            progress = i / (num_frames - 1)
            prompt = f"A beautiful sunset over a cityscape, golden hour, cinematic lighting, progress: {progress:.2f}"
            image = pipeline(prompt).images[0]
            frames.append(image)

        # Save frames and create a video (requires opencv-python)
        height, width = frames[0].size
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        video_writer = cv2.VideoWriter('sunset_animation.avi', fourcc, num_frames / duration_seconds, (width, height))

        temp_dir = "temp_frames"
        os.makedirs(temp_dir, exist_ok=True)
        filepaths = []
        for i, frame in enumerate(frames):
            filepath = os.path.join(temp_dir, f"frame_{i}.png")
            frame.save(filepath)
            filepaths.append(filepath)
            img_cv = cv2.cvtColor(np.array(frame), cv2.COLOR_RGB2BGR)
            video_writer.write(img_cv)

        video_writer.release()
        import shutil
        shutil.rmtree(temp_dir)
        return 'sunset_animation.avi'

    except Exception as e:
        return f"Error generating animation: {e}. Make sure you have a GPU runtime in Colab for better performance."

def apply_color_grade(input_video_path: str):
    """Applies a simple color grade (e.g., making it warmer) to a video."""
    try:
        cap = cv2.VideoCapture(input_video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        output_path = 'color_graded_video.avi'
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            # Simple warm color grade: increase red and green slightly
            frame[:, :, 2] = np.clip(frame[:, :, 2] * 1.1, 0, 255) # Red
            frame[:, :, 1] = np.clip(frame[:, :, 1] * 1.05, 0, 255) # Green
            out.write(frame)

        cap.release()
        out.release()
        return output_path
    except Exception as e:
        return f"Error applying color grade: {e}"


In [ ]:
# --- 4. User Interface with Gradio ---

def main():
    with gr.Blocks() as iface:
        gr.Markdown("# AI Content Generator for Games, Music, and Videos")

        with gr.Tab("Game Content"):
            gr.Markdown("## Game Asset Generation")
            asset_prompt = gr.Textbox(label="Enter prompt for game asset (e.g., a fantasy castle, a cyberpunk character)")
            asset_output = gr.Image(label="Generated Asset")
            asset_button = gr.Button("Generate Asset")
            asset_button.click(generate_game_asset, inputs=asset_prompt, outputs=asset_output)

            gr.Markdown("## Procedural Level Generation")
            with gr.Row():
                level_width = gr.Slider(minimum=16, maximum=640, step=8, value=32, label="Level Width")
                level_height = gr.Slider(minimum=16, maximum=640, step=8, value=32, label="Level Height")
                level_density = gr.Slider(minimum=0.1, maximum=0.9, step=0.1, value=0.4, label="Initial Density")
                level_iterations = gr.Slider(minimum=1, maximum=10, step=1, value=5, label="Iterations")
            level_output = gr.Image(label="Generated Level")
            level_button = gr.Button("Generate Level")
            level_button.click(generate_procedural_level, inputs=[level_width, level_height, level_density, level_iterations], outputs=level_output)

        with gr.Tab("Music Composition"):
            gr.Markdown("## Melody Generation")
            melody_genre = gr.Dropdown(["Ambient", "Pop", "Classical"], label="Genre")
            melody_tempo = gr.Slider(minimum=60, maximum=180, step=10, value=120, label="Tempo (BPM)")
            melody_length = gr.Slider(minimum=5, maximum=30, step=5, value=10, label="Length (seconds)")
            melody_output = gr.Audio(label="Generated Melody")
            melody_button = gr.Button("Generate Melody")
            melody_button.click(lambda genre, tempo, length: save_audio(*generate_melody(genre, tempo, length)),
                                inputs=[melody_genre, melody_tempo, melody_length], outputs=melody_output)

            gr.Markdown("## Beat Creation")
            beat_tempo = gr.Slider(minimum=60, maximum=180, step=10, value=120, label="Tempo (BPM)")
            beat_length = gr.Slider(minimum=5, maximum=30, step=5, value=10, label="Length (seconds)")
            beat_output = gr.Audio(label="Generated Beat")
            beat_button = gr.Button("Generate Beat")
            beat_button.click(lambda tempo, length: save_audio(*generate_beat(tempo, length)),
                              inputs=[beat_tempo, beat_length], outputs=beat_output)

        with gr.Tab("Video Synthesis"):
            gr.Markdown("## Sunset Animation")
            animation_duration = gr.Slider(minimum=5, maximum=20, step=5, value=10, label="Duration (seconds)")
            animation_frames = gr.Slider(minimum=15, maximum=60, step=15, value=30, label="Number of Frames")

            animation_output = gr.Video(label="Generated Sunset Animation")
            animation_button = gr.Button("Generate Sunset Animation")
            animation_button.click(generate_sunset_animation, inputs=[animation_duration, animation_frames], outputs=animation_output)

            gr.Markdown("## Apply Color Grade")
            color_grade_input = gr.Video(label="Upload Video to Color Grade")
            color_grade_output = gr.Video(label="Color Graded Video")
            color_grade_button = gr.Button("Apply Warm Color Grade")
            color_grade_button.click(apply_color_grade, inputs=color_grade_input, outputs=color_grade_output)

        gr.Markdown("## Learning Objectives")
        gr.Markdown("""
        - **Diffusion Models (e.g., Stable Diffusion):** Used for generating realistic images by learning to reverse a noise process. You provide a text prompt, and the model generates an image that matches the description.
        - **Generative Adversarial Networks (GANs):** While not explicitly used in this simplified example, GANs are another powerful type of generative model. They consist of two neural networks, a generator and a discriminator, that compete with each other to produce realistic data.
        - **Procedural Generation:** Algorithms used to create content algorithmically rather than manually. Cellular automata, as used for level generation, are simple rules applied to a grid that can result in complex patterns.
        """)

        gr.Markdown("## Ethical Considerations")
        gr.Markdown("""
        - **Copyright:** Be aware of the copyright implications of using AI-generated content, especially if using pre-trained models.
        - **Responsible Use:** Use these tools responsibly and ethically. Avoid generating harmful or biased content.
        """)

        gr.Markdown("## Example Outputs")
        gr.Markdown("""
        - **Game Content:** A generated image of a "fantasy character" or a black and white image representing a procedural dungeon layout.
        - **Music:** A short sine wave melody or a basic kick drum beat. (Note: These are simplified examples. Real AI music generation is much more sophisticated).
        - **Video:** A short animation of a sunset fading into night over a cityscape, or a video with slightly warmer colors.
        """)

    iface.launch(debug = True)


In [ ]:
if __name__ == "__main__":
    main()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f0b5747d50bdaefe93.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/gradio/components/video.py:351: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/gradio/components/video.py:351: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f0b5747d50bdaefe93.gradio.live
